# Runnable report workflow

This notebook calls the maintained package: preparation, real report-width LSTM/GRU and regression training, validation selection, held-out evaluation after loading, and one-step inference. The two-epoch run is an execution smoke test on committed daily sums, not a reproduction of published minute/hourly scores. Run from the repository root or LSTM directory after installing requirements plus Jupyter. See [coverage](../REPORT_COVERAGE.md).

In [ ]:
from pathlib import Path
import sys
from datetime import datetime

ROOT = Path.cwd()
if not (ROOT / "load_forecasting").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "load_forecasting").is_dir(), "Open notebook from repository root or LSTM directory"
sys.path.insert(0, str(ROOT))
from load_forecasting.config import Config
from load_forecasting.data import prepare_csv
from load_forecasting.workflow import train, evaluate, predict

run = ROOT / "runs" / ("notebook-" + datetime.now().strftime("%Y%m%d-%H%M%S-%f"))
run.mkdir(parents=True)
prepared = run / "daily.csv"
frame = prepare_csv(ROOT / "LSTM" / "household_daily.csv", prepared)
print(len(frame), "daily rows; target units: sum of minute kW samples")
frame.head()

Chronological 70/15/15 split by target time; no future observations in each input window. Training-only preprocessing is saved. Epochs are selected on validation loss. Test rows do not select any model state. Daily source values are retained without an unsupported unit conversion.

In [ ]:
config = Config.read(ROOT / "configs" / "smoke.json")
model_dir = run / "models"
scores = train(prepared, model_dir, config)
scores["validation"]

In [ ]:
loaded_test_scores = evaluate(model_dir, prepared)
assert loaded_test_scores == scores["test"]
loaded_test_scores

In [ ]:
forecast = predict(model_dir, prepared)
forecast

The forecast uses the last observed window and stored preprocessing/weights. To train longer, use configs/daily.json (10 maximum epochs) in a fresh run. Original raw minute data is required for configs/lstm_hourly.json and configs/gru_minute.json. The historical report scores are unverified; see REPORT_COVERAGE.md.